# Train Random Forest

# set env and path

In [1]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATA_DIR = PROJECT_ROOT/ "data"
PROCESS_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT/ "reports"
SRC_DIR = PROJECT_ROOT / "src"


sys.path.insert(0, str(SRC_DIR))

print(PROJECT_ROOT, SRC_DIR)


D:\HCMUT\ML\BTL\ml-btl-project D:\HCMUT\ML\BTL\ml-btl-project\src


# Load data

In [2]:
import pandas as pd

from preprocess import load_processed_data, split_features_target, TARGET_COL

train_df, test_df = load_processed_data()

print(f"train df: {train_df.shape}")
print(f"test df: {test_df.shape}")
print(f"Target col: {TARGET_COL}")


x_train, y_train = split_features_target(train_df)
x_test, y_test = split_features_target(test_df)

print(f"\nX_train: {x_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {x_test.shape}  | y_test : {y_test.shape}")




train df: (800, 19)
test df: (200, 19)
Target col: credit_risk

X_train: (800, 18) | y_train: (800,)
X_test : (200, 18)  | y_test : (200,)


In [3]:
print(f"tỷ lệ default (y=1) y_train: {y_train.mean():.4f}")
print(f"tỷ lệ default (y=1) y_test: {y_test.mean():.4f}")
display(x_train.head(10))

tỷ lệ default (y=1) y_train: 0.3000
tỷ lệ default (y=1) y_test: 0.3000


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,people_liable,foreign_worker
0,no checking account,30,critical account/other credits existing,domestic appliances,4530,... < 100 DM,4 <= ... < 7 years,4,female : divorced/separated/married,none,4,car or other,26,none,rent,1,1,yes
1,0 <= ... < 200 DM,30,delay in paying off in the past,others,2503,100 <= ... < 500 DM,... >= 7 years,4,male : single,none,2,building society savings agreement/life insurance,41,stores,own,2,1,yes
2,0 <= ... < 200 DM,12,existing credits paid back duly till now,domestic appliances,1567,... < 100 DM,1 <= ... < 4 years,1,female : divorced/separated/married,none,1,car or other,22,none,own,1,1,yes
3,0 <= ... < 200 DM,21,existing credits paid back duly till now,radio/television,3976,unknown/no savings account,4 <= ... < 7 years,2,male : single,none,3,car or other,35,none,own,1,1,yes
4,no checking account,9,existing credits paid back duly till now,radio/television,2301,100 <= ... < 500 DM,... < 1 year,2,female : divorced/separated/married,none,4,building society savings agreement/life insurance,22,none,rent,1,1,yes
5,... < 100 DM,12,existing credits paid back duly till now,car (new),1372,... < 100 DM,4 <= ... < 7 years,2,male : divorced/separated,none,3,car or other,36,none,own,1,1,yes
6,no checking account,24,all credits at this bank paid back duly,others,1559,... < 100 DM,4 <= ... < 7 years,4,male : single,none,4,car or other,50,bank,own,1,1,yes
7,... < 100 DM,6,existing credits paid back duly till now,radio/television,428,... < 100 DM,... >= 7 years,2,female : divorced/separated/married,none,1,building society savings agreement/life insurance,49,bank,own,1,1,yes
8,no checking account,21,delay in paying off in the past,car (used),2993,... < 100 DM,1 <= ... < 4 years,3,male : single,none,2,real estate,28,stores,own,2,1,yes
9,... < 100 DM,20,existing credits paid back duly till now,radio/television,2212,unknown/no savings account,4 <= ... < 7 years,4,male : single,none,4,car or other,39,none,own,1,1,yes


# Build preprocessor & pipeline baseline

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from preprocess import build_preprocessor
RANDOM_STATE = 0


preprocessor = build_preprocessor(x_train, scale_numeric=False)

rf_baseline = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline_baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf_baseline),
])

print(pipeline_baseline)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['duration', 'amount',
                                                   'installment_rate',
                                                   'present_residence', 'age',
                                                   'number_credits',
                                                   'people_liable']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['status', 'credit_history',
                                                   'purpose', 'savings',
                                                   'employment_duration',
                                                   'personal_status_sex',
                                                   'other_debtors', 'property',
      

# Train baseline

In [5]:
import time
import numpy as np

from sklearn.metrics import classification_report, confusion_matrix

from evaluate import calculate_metrics


t0 = time.perf_counter()
pipeline_baseline.fit(x_train, y_train)

fit_time = time.perf_counter() - t0



## Evaluate baseline

In [6]:
threshold = 0.5
base_metrics = calculate_metrics(pipeline_baseline, x_test, y_test, threshold=threshold)

y_proba_base = pipeline_baseline.predict_proba(x_test)[:,1]
y_pred_base = (y_proba_base >= threshold).astype(int)

print("========Baseline=========")
print(classification_report(y_test, y_pred_base, zero_division=0))

print("========Metrics=========")
for k, v in base_metrics.items():
    print(f"{k:15s}: {v:.4f}")

print("========Confusion Matrix=========")
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_base, labels=[0, 1]).ravel()
print(f"TN = {tn} | FP = {fp}")
print(f"FN = {fn} | TP = {tp}")

print(f"Fit time baseline: {fit_time:.2f}s")

========Baseline=========
              precision    recall  f1-score   support

           0       0.80      0.79      0.79       140
           1       0.52      0.55      0.54        60

    accuracy                           0.71       200
   macro avg       0.66      0.67      0.67       200
weighted avg       0.72      0.71      0.72       200

========Metrics=========
Precision      : 0.5238
Recall         : 0.5500
F1             : 0.5366
Expected Cost  : 0.8250
ROC-AUC        : 0.7757
PR-AUC         : 0.6154
========Confusion Matrix=========
TN = 110 | FP = 30
FN = 27 | TP = 33
Fit time baseline: 0.38s


# Tuning GridSearchCV (average_precision)


## Set grid + cv

In [7]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    "model__n_estimators":      [200, 300, 500],
    "model__max_depth":         [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features":      ["sqrt", "log2"],
}

# CV 5-fold 
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# GridSearchCV với tune average_precision (PR-AUC)
grid = GridSearchCV(
    estimator=pipeline_baseline,
    param_grid=param_grid,
    scoring="average_precision",  
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

## Fit model

In [8]:
print("Đang tuning ...")
t0 = time.perf_counter()
grid.fit(x_train, y_train)
tuning_time = time.perf_counter() - t0
print(f"Tuning xong. Tuning time: {tuning_time:.2f}s")


print(f"\nBest params        : {grid.best_params_}")
print(f"Best CV PR-AUC     : {grid.best_score_:.4f}")
print(f"Best estimator     : {grid.best_estimator_.named_steps['model']}")
print(f"Tuning time        : {tuning_time:.2f}s")

Đang tuning ...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Tuning xong. Tuning time: 45.56s

Best params        : {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_split': 2, 'model__n_estimators': 500}
Best CV PR-AUC     : 0.6532
Best estimator     : RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=500,
                       n_jobs=-1, random_state=0)
Tuning time        : 45.56s


## Evaluate best model

In [9]:
best_tuned = grid.best_estimator_


threshold = 0.5
tuned_metrics = calculate_metrics(best_tuned, x_test, y_test, threshold=threshold)

y_proba_tuned = best_tuned.predict_proba(x_test)[:,1]
y_pred_tuned = (y_proba_tuned >= threshold).astype(int)

print("========Tuned =========")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

print("========Metrics=========")
for k, v in tuned_metrics.items():
    print(f"{k:15s}: {v:.4f}")

print("========Confusion Matrix=========")
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_tuned, labels=[0, 1]).ravel()
print(f"TN = {tn} | FP = {fp}")
print(f"FN = {fn} | TP = {tp}")

print(f"Fit tuning time : {tuning_time:.2f}s")

========Tuned =========
              precision    recall  f1-score   support

           0       0.84      0.74      0.79       140
           1       0.53      0.67      0.59        60

    accuracy                           0.72       200
   macro avg       0.68      0.70      0.69       200
weighted avg       0.74      0.72      0.73       200

========Metrics=========
Precision      : 0.5263
Recall         : 0.6667
F1             : 0.5882
Expected Cost  : 0.6800
ROC-AUC        : 0.7818
PR-AUC         : 0.6366
========Confusion Matrix=========
TN = 104 | FP = 36
FN = 20 | TP = 40
Fit tuning time : 45.56s


# save model

In [10]:
import joblib



# 6. save model + result

joblib.dump(pipeline_baseline, MODEL_DIR / "rf_baseline.pkl")
joblib.dump(best_tuned,        MODEL_DIR / "rf_tuned.pkl")


best_params_str = str(grid.best_params_)
pd.DataFrame({"RF": [best_params_str]}).to_csv(
    REPORT_DIR / "rf_best_params.csv", index=False
)

results_df = pd.DataFrame({
    "RF Baseline @0.5": base_metrics,
    "RF Tuned @0.5":    tuned_metrics,
})

results_df.loc["Fit time baseline (s)"] = [fit_time, np.nan]
results_df.loc["Tuning time (s)"]       = [np.nan, tuning_time]

results_df = results_df.round(4)

print("\n=== SUMMARY (RF) ===")
print(results_df.to_string())

results_df.to_csv(REPORT_DIR / "rf_results.csv")



=== SUMMARY (RF) ===
                       RF Baseline @0.5  RF Tuned @0.5
Precision                        0.5238         0.5263
Recall                           0.5500         0.6667
F1                               0.5366         0.5882
Expected Cost                    0.8250         0.6800
ROC-AUC                          0.7757         0.7818
PR-AUC                           0.6154         0.6366
Fit time baseline (s)            0.3782            NaN
Tuning time (s)                     NaN        45.5586
